In [68]:
import sys
sys.executable

'c:\\Python311\\python.exe'

In [69]:
import pandas as pd 
import os 



In [70]:
df = pd.read_csv("train.csv")
df.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [71]:
text_column = "comment_text"
label_columns = ["toxic", "severe_toxic", "obscene", "threat","insult", "identity_hate"]

df[text_column].head()

0    Explanation\nWhy the edits made under my usern...
1    D'aww! He matches this background colour I'm s...
2    Hey man, I'm really not trying to edit war. It...
3    "\nMore\nI can't make any real suggestions on ...
4    You, sir, are my hero. Any chance you remember...
Name: comment_text, dtype: object

In [72]:
df.shape

(159571, 8)

In [73]:
df[label_columns].sum().sort_values(ascending=False)

toxic            15294
obscene           8449
insult            7877
severe_toxic      1595
identity_hate     1405
threat             478
dtype: int64

In [74]:
df["clean"] = (df[label_columns].sum(axis=1)==0)
df["clean"].value_counts()

clean
True     143346
False     16225
Name: count, dtype: int64

In [75]:
df[label_columns].sum(axis=1).value_counts().head()

0    143346
1      6360
3      4209
2      3480
4      1760
Name: count, dtype: int64

# Baseline model

In [76]:
X = df["comment_text"]
Y = df[label_columns]

In [77]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.2,
    random_state=42
)


In [78]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1,2)
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)


In [79]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

model = OneVsRestClassifier(
    LogisticRegression(max_iter=1000)
)

model.fit(X_train_tfidf,Y_train)

,estimator,LogisticRegre...max_iter=1000)
,n_jobs,None
,verbose,0
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None


In [80]:
y_pred = model.predict(X_test_tfidf)

In [81]:
from sklearn.metrics import classification_report

print(classification_report(Y_test, y_pred, target_names=label_columns))

               precision    recall  f1-score   support

        toxic       0.92      0.61      0.73      3056
 severe_toxic       0.59      0.21      0.30       321
      obscene       0.92      0.60      0.72      1715
       threat       0.47      0.09      0.16        74
       insult       0.85      0.51      0.64      1614
identity_hate       0.75      0.15      0.25       294

    micro avg       0.89      0.54      0.67      7074
    macro avg       0.75      0.36      0.47      7074
 weighted avg       0.88      0.54      0.66      7074
  samples avg       0.06      0.05      0.05      7074



c:\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


here normal logistic regression and TF-IDF (baseline model) fails due to small represtation of some label in dataset and not considerring the context of word used. thus we use BERT so model can take decision based on the context of the comment also.
 

In [16]:
import pandas as pd
import numpy as np
import torch

from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

In [17]:
df = pd.read_csv("train.csv")

df.head()


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [18]:
labels = df.columns[2:]

df['labels'] = df[labels].values.tolist()



In [19]:
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)

In [20]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

c:\Python311\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [21]:
train_encodings = tokenizer(
    train_df["comment_text"].tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

val_encodings = tokenizer(
    val_df["comment_text"].tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

In [22]:
class ToxicDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):

        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx]).float()

        return item

    def __len__(self):
        return len(self.labels)

In [23]:
train_dataset = ToxicDataset(train_encodings, train_df["labels"].tolist())
val_dataset = ToxicDataset(val_encodings, val_df["labels"].tolist())



In [24]:
print(len(train_dataset))
print(len(val_dataset))

143613
15958


In [25]:
from torch.utils.data import Subset

In [26]:
train_dataset = Subset(train_dataset, range(20000))
val_dataset = Subset(val_dataset, range(5000))

In [27]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=6,
    problem_type="multi_label_classification"
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [28]:
import sys
print(sys.executable)

c:\Python311\python.exe


In [29]:
!pip install accelerate


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
import accelerate
print(accelerate.__version__)

0.27.2


In [31]:
import accelerate
import transformers
import torch

print("Accelerate:", accelerate.__version__)
print("Transformers:", transformers.__version__)
print("Torch:", torch.__version__)

Accelerate: 0.27.2
Transformers: 4.38.2
Torch: 2.5.1+cu121


In [32]:
from transformers import TrainingArguments

In [33]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100
)

In [34]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=6
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [35]:
print(type(train_dataset))

print("df" in globals())
print("train_dataset" in globals())
print("val_dataset" in globals())
print("model" in globals())

<class 'torch.utils.data.dataset.Subset'>
True
True
True
True


In [36]:
!pip install transformers==4.38.2
!pip install accelerate==0.27.2


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [37]:
from transformers import Trainer

In [38]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=val_dataset
)

In [39]:
print(len(train_dataset))
print(len(val_dataset))

20000
5000


In [40]:
print(training_args.num_train_epochs)

1


In [41]:
trainer.train()

  0%|          | 0/2500 [00:00<?, ?it/s]

{'loss': 0.2658, 'grad_norm': 0.46125972270965576, 'learning_rate': 1.9200000000000003e-05, 'epoch': 0.04}
{'loss': 0.0995, 'grad_norm': 0.3950829803943634, 'learning_rate': 1.8400000000000003e-05, 'epoch': 0.08}
{'loss': 0.0851, 'grad_norm': 0.3973466455936432, 'learning_rate': 1.76e-05, 'epoch': 0.12}
{'loss': 0.0793, 'grad_norm': 1.4242699146270752, 'learning_rate': 1.6800000000000002e-05, 'epoch': 0.16}
{'loss': 0.0533, 'grad_norm': 2.1021502017974854, 'learning_rate': 1.6000000000000003e-05, 'epoch': 0.2}
{'loss': 0.0626, 'grad_norm': 3.178443431854248, 'learning_rate': 1.5200000000000002e-05, 'epoch': 0.24}
{'loss': 0.0677, 'grad_norm': 1.8550087213516235, 'learning_rate': 1.4400000000000001e-05, 'epoch': 0.28}
{'loss': 0.0541, 'grad_norm': 0.11581229418516159, 'learning_rate': 1.3600000000000002e-05, 'epoch': 0.32}
{'loss': 0.0648, 'grad_norm': 1.9780396223068237, 'learning_rate': 1.2800000000000001e-05, 'epoch': 0.36}
{'loss': 0.0489, 'grad_norm': 0.07879596203565598, 'learning

  0%|          | 0/625 [00:00<?, ?it/s]

Checkpoint destination directory ./results\checkpoint-2500 already exists and is non-empty. Saving will proceed but saved results may be invalid.


{'eval_loss': 0.043868232518434525, 'eval_runtime': 26.3688, 'eval_samples_per_second': 189.618, 'eval_steps_per_second': 23.702, 'epoch': 1.0}
{'train_runtime': 467.3754, 'train_samples_per_second': 42.792, 'train_steps_per_second': 5.349, 'train_loss': 0.06572113151550293, 'epoch': 1.0}


TrainOutput(global_step=2500, training_loss=0.06572113151550293, metrics={'train_runtime': 467.3754, 'train_samples_per_second': 42.792, 'train_steps_per_second': 5.349, 'train_loss': 0.06572113151550293, 'epoch': 1.0})

In [42]:
predictions = trainer.predict(val_dataset)

  0%|          | 0/625 [00:00<?, ?it/s]

In [43]:
import torch

logits = predictions.predictions
probs = torch.sigmoid(torch.tensor(logits))

In [44]:
y_pred = (probs > 0.5).int().numpy()

In [45]:
y_true = val_df[labels].values

In [46]:
preds = trainer.predict(val_dataset)

  0%|          | 0/625 [00:00<?, ?it/s]

In [47]:
y_pred = preds.predictions
y_true = preds.label_ids

In [48]:
from scipy.special import expit
import numpy as np

y_pred = expit(y_pred)
y_pred = (y_pred > 0.5).astype(int)

In [49]:
from sklearn.metrics import f1_score

f1_micro = f1_score(y_true, y_pred, average="micro")
f1_macro = f1_score(y_true, y_pred, average="macro")

print("Micro F1:", f1_micro)
print("Macro F1:", f1_macro)

Micro F1: 0.7585858585858586
Macro F1: 0.3965613128441887


In [50]:
trainer.save_model("toxic_model")
tokenizer.save_pretrained("toxic_model")

('toxic_model\\tokenizer_config.json',
 'toxic_model\\special_tokens_map.json',
 'toxic_model\\vocab.txt',
 'toxic_model\\added_tokens.json',
 'toxic_model\\tokenizer.json')

In [56]:
import torch
from scipy.special import expit

labels = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

def predict_toxicity(text):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    # Move inputs to the same device as the model
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.sigmoid(outputs.logits).cpu().numpy()[0]

    predictions = (probs > 0.5).astype(int)

    results = {
        label: {
            "probability": float(prob),
            "prediction": int(pred)
        }
        for label, prob, pred in zip(label_columns, probs, predictions)
    }

    return results

In [59]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

Using device: cuda


In [60]:
predict_toxicity("You are an idiot and I hate you")

{'toxic': {'probability': 0.9572539925575256, 'prediction': 1},
 'severe_toxic': {'probability': 0.16608035564422607, 'prediction': 0},
 'obscene': {'probability': 0.803930938243866, 'prediction': 1},
 'threat': {'probability': 0.05760341137647629, 'prediction': 0},
 'insult': {'probability': 0.8521487712860107, 'prediction': 1},
 'identity_hate': {'probability': 0.18146489560604095, 'prediction': 0}}

In [62]:
predict_toxicity("You are an idiot and I hate you")

{'toxic': {'probability': 0.9572539925575256, 'prediction': 1},
 'severe_toxic': {'probability': 0.16608035564422607, 'prediction': 0},
 'obscene': {'probability': 0.803930938243866, 'prediction': 1},
 'threat': {'probability': 0.05760341137647629, 'prediction': 0},
 'insult': {'probability': 0.8521487712860107, 'prediction': 1},
 'identity_hate': {'probability': 0.18146489560604095, 'prediction': 0}}

In [63]:

predict_toxicity("Thank you for your help, I really appreciate it.")

{'toxic': {'probability': 0.0033207263331860304, 'prediction': 0},
 'severe_toxic': {'probability': 0.0005374779575504363, 'prediction': 0},
 'obscene': {'probability': 0.0015094754053279757, 'prediction': 0},
 'threat': {'probability': 0.0007419536705128849, 'prediction': 0},
 'insult': {'probability': 0.0012505297781899571, 'prediction': 0},
 'identity_hate': {'probability': 0.000643993669655174, 'prediction': 0}}

In [64]:
predict_toxicity("I strongly disagree with your argument. Your explanation is incorrect.")

{'toxic': {'probability': 0.0036093262024223804, 'prediction': 0},
 'severe_toxic': {'probability': 0.00048457045340910554, 'prediction': 0},
 'obscene': {'probability': 0.001356468303129077, 'prediction': 0},
 'threat': {'probability': 0.0006828818586654961, 'prediction': 0},
 'insult': {'probability': 0.0013240313855931163, 'prediction': 0},
 'identity_hate': {'probability': 0.0006416497053578496, 'prediction': 0}}